In [ ]:
import pandas as pd
import requests

# Wikipedia API - get the U.S. Open (golf) page content
wiki_api = "https://en.wikipedia.org/w/api.php"

params = {
    "action": "parse",
    "page": "U.S._Open_(golf)",
    "format": "json",
    "prop": "text"
}

response = requests.get(wiki_api, params=params)
data = response.json()
print(f"Status: {response.status_code}")

In [ ]:
# Parse tables from the HTML content
from io import StringIO

html_content = data['parse']['text']['*']

# Use pandas to extract all tables
dfs = pd.read_html(StringIO(html_content))
print(f"Found {len(dfs)} table(s)")

# Preview each table
for i, df in enumerate(dfs[:10]):  # First 10 tables
    print(f"\n=== Table {i} ===")
    print(df.columns.tolist())
    print(df.head(3))

In [ ]:
# Find the Winners table (has Year, Venue, Winner columns)
# Usually one of the larger tables with historical data
winners_df = None

for i, df in enumerate(dfs):
    cols = [str(c).lower() for c in df.columns.tolist()]
    # Look for table with venue/course data
    if any('venue' in c or 'location' in c for c in cols):
        print(f"Found venue table at index {i}")
        winners_df = df
        break
    if any('year' in c for c in cols) and len(df) > 50:
        print(f"Found large historical table at index {i}")
        winners_df = df

if winners_df is not None:
    print(f"\nColumns: {winners_df.columns.tolist()}")
    winners_df.head(20)

In [ ]:
# Extract unique courses/venues from the winners table
if winners_df is not None:
    # Find the venue column (might be called Venue, Location, Course, etc.)
    venue_col = None
    for col in winners_df.columns:
        if any(x in str(col).lower() for x in ['venue', 'location', 'course']):
            venue_col = col
            break
    
    if venue_col:
        courses = winners_df[venue_col].dropna().unique()
        print(f"Found {len(courses)} unique venues:\n")
        for course in sorted(courses):
            print(f"  - {course}")

In [ ]:
# Look for "Summary by course" table - has course frequency data
for i, df in enumerate(dfs):
    # Check if this looks like the summary table
    if len(df) > 5 and len(df) < 60:
        first_col = str(df.columns[0]).lower()
        if 'course' in first_col or 'venue' in first_col:
            print(f"=== Possible Summary Table (index {i}) ===")
            print(df)
            print("\n")

In [ ]:
# Get links from the page (to find individual course Wikipedia pages)
params_links = {
    "action": "parse",
    "page": "U.S._Open_(golf)",
    "format": "json",
    "prop": "links"
}

response = requests.get(wiki_api, params=params_links)
links_data = response.json()

# Filter for golf club/course links
golf_links = []
for link in links_data['parse']['links']:
    title = link.get('*', '')
    if any(x in title.lower() for x in ['golf club', 'country club', 'golf course', 'golf links']):
        golf_links.append(title)

print(f"Found {len(golf_links)} golf course links:\n")
for link in sorted(set(golf_links)):
    print(f"  - {link}")